# 02 — Gold | Player Performance KPIs

**Project:** MLB Performance Intelligence Platform

Season lines and game-by-game form for every hitter and pitcher (KPI catalog BAT-001 … BAT-010,
PIT-001 … PIT-010, FORM-001 … FORM-003).

| Table | Grain | Highlights |
|---|---|---|
| `gold.player_season_batting` | player × season × season type | AVG, OBP, SLG, OPS, ISO, BABIP, K%, BB%, batting-title qualification, league ranks |
| `gold.player_season_pitching` | player × season × season type | ERA, WHIP, K/9, BB/9, HR/9, K-BB%, **FIP** (league constant per season), quality starts, role |
| `gold.batting_game_log` | player × game | Rolling last-15-games AVG/OBP/SLG/OPS and season-to-date line |
| `gold.pitching_game_log` | pitcher × appearance | Rolling last-5-appearances ERA/WHIP and season-to-date ERA |

**Formulas** (MLB / FanGraphs definitions)
- OBP = (H + BB + HBP) / (AB + BB + HBP + SF) · SLG = TB / AB · OPS = OBP + SLG · ISO = SLG − AVG
- BABIP = (H − HR) / (AB − SO − HR + SF)
- ERA = 9 × ER / IP · WHIP = (BB + H) / IP · FIP = (13·HR + 3·(BB + HBP) − 2·SO) / IP + cFIP, where cFIP makes league FIP equal league ERA for the season
- Qualified: PA ≥ 3.1 × team games (hitters), IP ≥ 1.0 × team games (pitchers), regular season only
- Age = age on June 30 of the season (Baseball-Reference convention)

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)

RUN_ID = P.new_run_id()
T = lambda s, n: C.table(s, n)

# Team games per season (for qualification thresholds): the most games any team has played
spark.sql(f"""
SELECT season, season_type, MAX(games) AS team_games
FROM (SELECT season, season_type, team_id, COUNT(*) AS games FROM {T('silver', 'fact_team_game')}
      WHERE is_completed GROUP BY ALL)
GROUP BY ALL""").createOrReplaceTempView("team_games")

## player_season_batting

In [ ]:
player_season_batting = spark.sql(f"""
WITH s AS (
  SELECT season, season_type, player_id,
    MAX_BY(team_id, official_date) AS last_team_id, COUNT(DISTINCT team_id) AS teams_played,
    COUNT(*) AS games, SUM(INT(is_starter)) AS games_started,
    SUM(pa) AS pa, SUM(ab) AS ab, SUM(r) AS r, SUM(h) AS h, SUM(singles) AS singles, SUM(doubles) AS doubles,
    SUM(triples) AS triples, SUM(hr) AS hr, SUM(rbi) AS rbi, SUM(bb) AS bb, SUM(ibb) AS ibb, SUM(so) AS so,
    SUM(hbp) AS hbp, SUM(sb) AS sb, SUM(cs) AS cs, SUM(sf) AS sf, SUM(sh) AS sh, SUM(tb) AS tb, SUM(gidp) AS gidp
  FROM {T('silver', 'fact_batting_game')} GROUP BY season, season_type, player_id
),
m AS (
  SELECT s.*,
    ROUND(h / NULLIF(ab, 0), 3) AS avg,
    ROUND((h + bb + hbp) / NULLIF(ab + bb + hbp + sf, 0), 3) AS obp,
    ROUND(tb / NULLIF(ab, 0), 3) AS slg,
    ROUND((h + bb + hbp) / NULLIF(ab + bb + hbp + sf, 0) + tb / NULLIF(ab, 0), 3) AS ops,
    ROUND(tb / NULLIF(ab, 0) - h / NULLIF(ab, 0), 3) AS iso,
    ROUND((h - hr) / NULLIF(ab - so - hr + sf, 0), 3) AS babip,
    ROUND(100 * so / NULLIF(pa, 0), 1) AS k_pct,
    ROUND(100 * bb / NULLIF(pa, 0), 1) AS bb_pct,
    ROUND(sb / NULLIF(sb + cs, 0), 3) AS sb_success_rate,
    (s.season_type = 'Regular Season' AND pa >= {C.QUALIFIED_PA_PER_GAME} * tg.team_games) AS is_qualified
  FROM s JOIN team_games tg USING (season, season_type)
)
SELECT m.season, m.season_type, m.player_id, p.player_name, p.birth_country, p.is_dominican, p.primary_position, p.bats,
  FLOOR(MONTHS_BETWEEN(MAKE_DATE(m.season, 6, 30), p.birth_date) / 12) AS age,
  m.last_team_id AS team_id, t.team_abbreviation, m.teams_played,
  m.games, m.games_started, m.pa, m.ab, m.r, m.h, m.singles, m.doubles, m.triples, m.hr, m.rbi, m.bb, m.ibb, m.so,
  m.hbp, m.sb, m.cs, m.sf, m.sh, m.tb, m.gidp,
  m.avg, m.obp, m.slg, m.ops, m.iso, m.babip, m.k_pct, m.bb_pct, m.sb_success_rate, m.is_qualified,
  CASE WHEN m.is_qualified THEN RANK() OVER (PARTITION BY m.season, m.season_type, m.is_qualified ORDER BY m.ops DESC) END AS ops_rank_qualified,
  CASE WHEN m.is_qualified THEN RANK() OVER (PARTITION BY m.season, m.season_type, m.is_qualified ORDER BY m.avg DESC) END AS avg_rank_qualified,
  RANK() OVER (PARTITION BY m.season, m.season_type ORDER BY m.hr DESC) AS hr_rank,
  RANK() OVER (PARTITION BY m.season, m.season_type ORDER BY m.rbi DESC) AS rbi_rank,
  RANK() OVER (PARTITION BY m.season, m.season_type ORDER BY m.sb DESC) AS sb_rank
FROM m
LEFT JOIN {T('silver', 'dim_player')} p ON p.player_id = m.player_id
LEFT JOIN {T('silver', 'dim_team')} t ON t.team_id = m.last_team_id
""")

with P.log_step(spark, RUN_ID, "gold", "player_season_batting") as step:
    step.records = P.replace_table(spark, player_season_batting, T("gold", "player_season_batting"),
        "Gold: batting season line per player, season and season type (AVG/OBP/SLG/OPS, ISO, BABIP, K%, BB%, qualification, ranks)")

## player_season_pitching

In [ ]:
player_season_pitching = spark.sql(f"""
WITH s AS (
  SELECT season, season_type, player_id,
    MAX_BY(team_id, official_date) AS last_team_id, COUNT(DISTINCT team_id) AS teams_played,
    COUNT(*) AS games, SUM(INT(is_starter)) AS games_started, SUM(w) AS w, SUM(l) AS l, SUM(sv) AS sv,
    SUM(svo) AS svo, SUM(hld) AS hld, SUM(bs) AS bs, SUM(cg) AS cg, SUM(sho) AS sho,
    SUM(INT(is_quality_start)) AS quality_starts, SUM(outs) AS outs, SUM(batters_faced) AS batters_faced,
    SUM(pitches) AS pitches, SUM(h) AS h, SUM(r) AS r, SUM(er) AS er, SUM(bb) AS bb, SUM(ibb) AS ibb,
    SUM(so) AS so, SUM(hr) AS hr, SUM(hbp) AS hbp
  FROM {T('silver', 'fact_pitching_game')} GROUP BY season, season_type, player_id
),
lg AS (  -- FIP constant per season and season type
  SELECT season, season_type,
    27 * SUM(er) / SUM(outs) - (13 * SUM(hr) + 3 * (SUM(bb) + SUM(hbp)) - 2 * SUM(so)) / (SUM(outs) / 3) AS cfip
  FROM s GROUP BY season, season_type
),
m AS (
  SELECT s.*, ROUND(outs / 3, 2) AS ip,
    CONCAT(CAST(FLOOR(outs / 3) AS STRING), '.', CAST(outs % 3 AS STRING)) AS ip_display,
    ROUND(27 * er / NULLIF(outs, 0), 2) AS era,
    ROUND(3 * (bb + h) / NULLIF(outs, 0), 2) AS whip,
    ROUND(27 * so / NULLIF(outs, 0), 2) AS k_per_9,
    ROUND(27 * bb / NULLIF(outs, 0), 2) AS bb_per_9,
    ROUND(27 * hr / NULLIF(outs, 0), 2) AS hr_per_9,
    ROUND(so / NULLIF(bb, 0), 2) AS k_bb_ratio,
    ROUND(100 * so / NULLIF(batters_faced, 0), 1) AS k_pct,
    ROUND(100 * bb / NULLIF(batters_faced, 0), 1) AS bb_pct,
    ROUND(100 * (so - bb) / NULLIF(batters_faced, 0), 1) AS k_minus_bb_pct,
    ROUND((13 * hr + 3 * (bb + hbp) - 2 * so) / NULLIF(outs / 3, 0) + lg.cfip, 2) AS fip,
    CASE WHEN games_started >= games / 2 THEN 'Starter' ELSE 'Reliever' END AS role,
    (s.season_type = 'Regular Season' AND outs / 3 >= {C.QUALIFIED_IP_PER_GAME} * tg.team_games) AS is_qualified
  FROM s JOIN lg USING (season, season_type) JOIN team_games tg USING (season, season_type)
)
SELECT m.season, m.season_type, m.player_id, p.player_name, p.birth_country, p.is_dominican, p.throws,
  FLOOR(MONTHS_BETWEEN(MAKE_DATE(m.season, 6, 30), p.birth_date) / 12) AS age,
  m.last_team_id AS team_id, t.team_abbreviation, m.teams_played, m.role,
  m.games, m.games_started, m.w, m.l, m.sv, m.svo, m.hld, m.bs, m.cg, m.sho, m.quality_starts,
  m.outs, m.ip, m.ip_display, m.batters_faced, m.pitches, m.h, m.r, m.er, m.bb, m.ibb, m.so, m.hr, m.hbp,
  m.era, m.whip, m.k_per_9, m.bb_per_9, m.hr_per_9, m.k_bb_ratio, m.k_pct, m.bb_pct, m.k_minus_bb_pct, m.fip,
  m.is_qualified,
  CASE WHEN m.is_qualified THEN RANK() OVER (PARTITION BY m.season, m.season_type, m.is_qualified ORDER BY m.era) END AS era_rank_qualified,
  RANK() OVER (PARTITION BY m.season, m.season_type ORDER BY m.so DESC) AS so_rank,
  RANK() OVER (PARTITION BY m.season, m.season_type ORDER BY m.w DESC) AS w_rank,
  RANK() OVER (PARTITION BY m.season, m.season_type ORDER BY m.sv DESC) AS sv_rank
FROM m
LEFT JOIN {T('silver', 'dim_player')} p ON p.player_id = m.player_id
LEFT JOIN {T('silver', 'dim_team')} t ON t.team_id = m.last_team_id
""")

with P.log_step(spark, RUN_ID, "gold", "player_season_pitching") as step:
    step.records = P.replace_table(spark, player_season_pitching, T("gold", "player_season_pitching"),
        "Gold: pitching season line per player, season and season type (ERA, WHIP, K/9, FIP, quality starts, qualification, ranks)")

## Form: rolling game logs

In [ ]:
batting_game_log = spark.sql(f"""
SELECT b.season, b.season_type, b.player_id, b.game_pk, b.official_date, b.team_id, b.opponent_id, b.is_home,
  b.lineup_slot, b.position, b.pa, b.ab, b.r, b.h, b.doubles, b.triples, b.hr, b.rbi, b.bb, b.so, b.sb,
  ROW_NUMBER() OVER w_cum AS player_game_number,
  ROUND(SUM(h) OVER w15 / NULLIF(SUM(ab) OVER w15, 0), 3) AS avg_last_15,
  ROUND((SUM(h + bb + hbp) OVER w15) / NULLIF(SUM(ab + bb + hbp + sf) OVER w15, 0)
        + (SUM(tb) OVER w15) / NULLIF(SUM(ab) OVER w15, 0), 3) AS ops_last_15,
  SUM(hr) OVER w15 AS hr_last_15,
  ROUND(SUM(h) OVER w_cum / NULLIF(SUM(ab) OVER w_cum, 0), 3) AS avg_season_to_date,
  ROUND((SUM(h + bb + hbp) OVER w_cum) / NULLIF(SUM(ab + bb + hbp + sf) OVER w_cum, 0)
        + (SUM(tb) OVER w_cum) / NULLIF(SUM(ab) OVER w_cum, 0), 3) AS ops_season_to_date,
  SUM(hr) OVER w_cum AS hr_season_to_date
FROM {T('silver', 'fact_batting_game')} b
WINDOW w_cum AS (PARTITION BY season, season_type, player_id ORDER BY official_date, game_pk ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
       w15   AS (PARTITION BY season, season_type, player_id ORDER BY official_date, game_pk ROWS BETWEEN 14 PRECEDING AND CURRENT ROW)
""")

pitching_game_log = spark.sql(f"""
SELECT season, season_type, player_id, game_pk, official_date, team_id, opponent_id, is_home, is_starter,
  innings_pitched_display, outs, h, er, bb, so, hr, pitches, decision, is_quality_start,
  ROW_NUMBER() OVER w_cum AS appearance_number,
  ROUND(27 * SUM(er) OVER w5 / NULLIF(SUM(outs) OVER w5, 0), 2) AS era_last_5,
  ROUND(3 * SUM(bb + h) OVER w5 / NULLIF(SUM(outs) OVER w5, 0), 2) AS whip_last_5,
  ROUND(27 * SUM(er) OVER w_cum / NULLIF(SUM(outs) OVER w_cum, 0), 2) AS era_season_to_date,
  SUM(so) OVER w_cum AS so_season_to_date
FROM {T('silver', 'fact_pitching_game')}
WINDOW w_cum AS (PARTITION BY season, season_type, player_id ORDER BY official_date, game_pk ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
       w5    AS (PARTITION BY season, season_type, player_id ORDER BY official_date, game_pk ROWS BETWEEN 4 PRECEDING AND CURRENT ROW)
""")

with P.log_step(spark, RUN_ID, "gold", "player_game_logs") as step:
    n1 = P.replace_table(spark, batting_game_log, T("gold", "batting_game_log"),
        "Gold: batting game log with rolling last-15-games and season-to-date AVG/OPS/HR")
    n2 = P.replace_table(spark, pitching_game_log, T("gold", "pitching_game_log"),
        "Gold: pitching game log with rolling last-5-appearances and season-to-date ERA/WHIP")
    step.records = n1 + n2

In [ ]:
display(spark.sql(f"""
    SELECT season, player_name, team_abbreviation, pa, avg, obp, slg, ops, hr, ops_rank_qualified
    FROM {T('gold', 'player_season_batting')}
    WHERE season_type = 'Regular Season' AND is_qualified ORDER BY season DESC, ops DESC LIMIT 10"""))